# Latent Diffusion UNet for Brain Tumor MRI Synthesis

This notebook implements the **diffusion UNet training pipeline** for a
latent diffusion model (LDM) applied to brain tumor MRI synthesis.

## Scope of this Notebook
- Defines the diffusion UNet architecture
- Implements the diffusion training loop
- Assumes the existence of:
  - A pretrained VAE (AutoencoderKL)
  - Preprocessed 2D MRI slices

❗ This notebook **does not train the VAE** and **does not include real MRI data**.
Those components are handled separately.


In [ ]:
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from tqdm import tqdm

from monai.data import Dataset
from monai.transforms import (
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityRanged,
    Compose,
)

from generative.networks.nets import DiffusionModelUNet
from generative.networks.schedulers import DDPMScheduler
from generative.inferers import LatentDiffusionInferer


## Diffusion UNet Architecture

The diffusion model operates in the **latent space** learned by a pretrained VAE.
This UNet predicts the noise added to latent representations at different
diffusion timesteps.

Key design choices:
- 2D convolutions (slice-based MRI training)
- Attention at deeper layers for global structure
- Input/output channels match VAE latent dimensionality


In [ ]:
def build_diffusion_unet(
    in_channels: int = 4,
    out_channels: int = 4,
):
    """
    Builds the Diffusion UNet used for latent diffusion.
    """
    model = DiffusionModelUNet(
        spatial_dims=2,
        in_channels=in_channels,
        out_channels=out_channels,
        num_res_blocks=2,
        num_channels=(128, 256, 512),
        attention_levels=(False, True, True),
        num_head_channels=(0, 256, 512),
    )
    return model


## Diffusion Training Pipeline

The diffusion model is trained using a DDPM objective in latent space.

Training steps per batch:
1. Encode MRI slice into latent space using a pretrained VAE
2. Add noise according to a randomly sampled timestep
3. Predict the noise using the diffusion UNet
4. Optimize mean-squared error between predicted and true noise

The VAE is frozen during diffusion training.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# External dependencies (to be provided by user)
vae = None                # pretrained AutoencoderKL
brats_slice_paths = None  # list of MRI slice paths
scale_factor = None       # latent scaling factor

if vae is None or brats_slice_paths is None or scale_factor is None:
    raise RuntimeError(
        "VAE, data paths, and scale factor must be provided "
        "before running diffusion training."
    )


In [ ]:
train_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    ScaleIntensityRanged(
        keys=["image"],
        a_min=0,
        a_max=300,
        b_min=-1.0,
        b_max=1.0,
        clip=True,
    ),
])

train_files = [{"image": p} for p in brats_slice_paths]

train_dataset = Dataset(train_files, transform=train_transforms)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
)


In [ ]:
unet = build_diffusion_unet().to(device)
scheduler = DDPMScheduler(num_train_timesteps=1000)
inferer = LatentDiffusionInferer(
    scheduler=scheduler,
    scale_factor=scale_factor,
)

optimizer = torch.optim.Adam(unet.parameters(), lr=1e-4)
scaler = GradScaler()

vae.eval()
vae.requires_grad_(False)

num_epochs = 20

for epoch in range(num_epochs):
    unet.train()
    epoch_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")

    for step, batch in enumerate(pbar):
        images = batch["image"].to(device)

        noise = torch.randn_like(images)
        timesteps = torch.randint(
            0, scheduler.num_train_timesteps,
            (images.shape[0],), device=device
        ).long()

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            noise_pred = inferer(
                inputs=images,
                diffusion_model=unet,
                noise=noise,
                timesteps=timesteps,
                autoencoder_model=vae,
            )
            loss = F.mse_loss(noise_pred, noise)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        pbar.set_postfix({"loss": epoch_loss / (step + 1)})

    print(f"Epoch {epoch} avg loss: {epoch_loss / (step + 1):.4f}")


## Notes

- This notebook intentionally excludes VAE training and MRI preprocessing.
- The diffusion model is designed to be modular and VAE-agnostic.
- Once trained, the diffusion UNet can be used to generate synthetic MRI
  slices by sampling from latent noise and decoding using the VAE.
